# Learned noise-aware attention — final 150k runs + single test evaluation

Continuation of the Colab screening. The variant was **selected on validation PSNR only** in
`learned_noise_screen_colab.ipynb`. This notebook trains that variant and a matched control from the
same pretrained LOL-v2-syn weights for 150k iterations, then evaluates both **once** on the held-out
100-image test set.

**Kaggle settings:** GPU T4 x2, Internet ON. Attach as input: the Colab `selection.json` (or paste
`SELECTED` below), and `learned_screen_state.zip` if you want to continue from the screened checkpoint
(optional — the final runs start from the official weights).

Expected time: ~13 h wall clock for the two runs in parallel on T4 x2 (~25 GPU-hours). Kaggle's
12 h session limit means use the timeout + resume pattern: Save Version, re-attach the state zip,
and rerun this notebook if needed.


In [ ]:
import os, sys, subprocess, glob, json, zipfile, shutil
from pathlib import Path

os.chdir('/kaggle/working')
if not Path('btp').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Crazycraftie/low-light-enhancement-btp.git','btp'],check=True)
os.chdir('/kaggle/working/btp')
print('HEAD:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
if not Path('baselines/Retinexformer').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/caiyuanhao1998/Retinexformer.git','baselines/Retinexformer'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','einops','gdown','lmdb','yacs','addict','future','natsort','yapf','thop','timm','lpips','scikit-image'],check=True)
import torch
assert torch.cuda.is_available(),'Enable GPU T4 x2'
OPT='baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml'


## Selection (validation only) — restore from Colab


In [ ]:
sels = glob.glob('/kaggle/input/**/selection.json', recursive=True)
if sels:
    sel = json.loads(Path(max(sels)).read_text())
    print('Colab selection:', sel)
    assert sel.get('best') in ('learned_s0.5','learned_s1.0'), 'Control won validation: stop and report negative result'
    SELECTED = sel['best']
    SCALE = 0.5 if SELECTED=='learned_s0.5' else 1.0
else:
    raise FileNotFoundError('Attach selection.json from the Colab screening as a Kaggle dataset input.')
print('SELECTED:', SELECTED, '| noise scale:', SCALE)


## Data + pretrained weights


In [ ]:
def gdl(fid,out,tries=4):
    out=Path(out); out.parent.mkdir(parents=True,exist_ok=True)
    for _ in range(tries):
        if subprocess.run(['gdown','-q',fid,'-O',str(out)]).returncode==0 and out.exists(): return
    raise RuntimeError('download failed '+str(out))

DATA=Path('data/LOLv2/Synthetic')
if not (DATA/'Train/Low').is_dir():
    attached=glob.glob('/kaggle/input/**/LOLv2/Synthetic/Train/Low',recursive=True)
    if attached:
        src=Path(attached[0]).parents[2]; Path('data').mkdir(exist_ok=True)
        if not Path('data/LOLv2').exists(): Path('data/LOLv2').symlink_to(src,target_is_directory=True)
    else:
        gdl('1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U','/kaggle/tmp/lolv2.zip')
        subprocess.run(['unzip','-q','-o','/kaggle/tmp/lolv2.zip','-d','data'],check=True)
assert len(list((DATA/'Test/Low').glob('*')))==100

WDIR=Path('baselines/Retinexformer/pretrained_weights')
c=list(WDIR.glob('**/LOL_v2_synthetic.pth')) if WDIR.exists() else []
if not c:
    subprocess.run(['gdown','--folder','1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV','-O','/kaggle/tmp/rf_weights'],check=True)
    c=list(Path('/kaggle/tmp/rf_weights').glob('**/LOL_v2_synthetic.pth'))
W=str(c[0]); print('weights:',W)


## Final matched training: control vs selected learned-noise variant

Both start from the same official weights, same seed, data, crops, LR schedule. Only `--learned-noise`
differs. Runs in parallel on GPU 0 / GPU 1. Timeout is 11 h (exit 124 => resume via state zip).


In [ ]:
RUNS=Path('runs/learned_final')
common=['--loss','l1','--iters','150000','--lr','2e-4','--batch','8','--crop','128','--seed','42',
        '--pretrained',W,'--opt',OPT,'--repo','baselines/Retinexformer','--data',str(DATA/'Train'),
        '--device','cuda','--val-fraction','0.1','--val-seed','2026','--val-every','5000',
        '--checkpoint-every','5000','--log-every','500']

def zip_state():
    p='/kaggle/working/learned_final_state.zip'
    with zipfile.ZipFile(p,'w',zipfile.ZIP_DEFLATED) as zf:
        for f in RUNS.rglob('*'):
            if f.is_file(): zf.write(f,f.as_posix())
    print('saved',p)

tasks=[('control',None),('learned_noise',SCALE)]
launched=[]
for gpu,(name,scale) in enumerate(tasks):
    out=RUNS/name; out.mkdir(parents=True,exist_ok=True)
    if (out/'metrics.json').exists():
        print('already complete:',out); continue
    cmd=['timeout','39600',sys.executable,'scripts/finetune.py']+common+['--out',str(out)]
    if scale is not None: cmd += ['--learned-noise','--noise-scale',str(scale)]
    if (out/'latest_state.pth').exists(): cmd += ['--resume']
    env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu)
    log=(out/'run.log').open('a')
    launched.append((name,subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env),log))
for name,proc,log in launched:
    rc=proc.wait(); log.close(); print(name,'exit',rc)
    if rc not in (0,124): raise RuntimeError((RUNS/name/'run.log').read_text()[-4000:])
zip_state()


## Single held-out test evaluation — only after both runs finish


In [ ]:
def infer(weights,inp,out,method,scale=None):
    cmd=[sys.executable,'scripts/retinexformer_infer.py','--repo','baselines/Retinexformer',
         '--opt',OPT,'--weights',str(weights),'--input',str(inp),'--output',str(out),
         '--method',method,'--device','cuda','--no-speed-log']
    if scale is not None: cmd += ['--learned-noise','--noise-scale',str(scale)]
    subprocess.run(cmd,check=True)

rows=[]
for name,scale in tasks:
    run=RUNS/name
    if not (run/'metrics.json').exists():
        print('incomplete, skipping:',name); continue
    w=run/'best.pth'
    if not w.exists(): w=run/'final.pth'
    method='LearnedNoise-control' if scale is None else f'LearnedNoise-a{SCALE}'
    out=Path('results/learned_noise')/method/'LOLv2-syn'
    infer(w,DATA/'Test/Low',out,method,scale)
    for script,extra in (
        ('scripts/evaluate.py',['--gt',str(DATA/'Test/Normal')]),
        ('scripts/eval_dark_regions.py',['--gt',str(DATA/'Test/Normal'),'--low',str(DATA/'Test/Low')])):
        subprocess.run([sys.executable,script,'--pred',str(out),'--method',method,'--dataset','LOLv2-syn']+extra,check=True)

with zipfile.ZipFile('/kaggle/working/learned_noise_results.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for p in (Path('results/results.csv'),Path('results/dark_region.csv')):
        if p.exists(): zf.write(p,p.name)
    for root in (Path('results/per_image'),Path('results/per_image_dark')):
        for p in root.rglob('*'):
            if p.is_file() and 'LearnedNoise' in str(p): zf.write(p,p.as_posix())
    for name in ('control','learned_noise'):
        for f in ('config.json','metrics.json','log.csv'):
            p=RUNS/name/f
            if p.exists(): zf.write(p,p.as_posix())
print('saved /kaggle/working/learned_noise_results.zip')
print('\n'.join(l for l in Path('results/results.csv').read_text().splitlines() if 'LearnedNoise' in l))


## Done

Download `learned_noise_results.zip`. If the selected variant beats the control on this single
held-out test with a paired sign test, we can legitimately report an improvement; otherwise the
paper reports a third controlled negative result. Send me the zip for the write-up.
